In [4]:
#Modelo Guía
%pip install -q "pulp[cbc]"

#Información Base del Problema
import pulp as lp
modelo = lp.LpProblem("Minimizacion_Costos_Variables_de_Produccion",lp.LpMinimize)
Meses = range(1, 7)

# Definición de Variables
P = {t: modelo.add_variable(f"P{t}", 0, None, lp.LpInteger)for t in Meses}
B = {t: modelo.add_variable(f"B{t}", 0, None, lp.LpInteger)for t in Meses}
I = {t: modelo.add_variable(f"I{t}", 0, None, lp.LpInteger)for t in Meses}
W = {t: modelo.add_variable(f"W{t}", 0, None, lp.LpInteger)for t in Meses}
H = {t: modelo.add_variable(f"H{t}", 0, None, lp.LpInteger)for t in Meses}
L = {t: modelo.add_variable(f"L{t}", 0, None, lp.LpInteger)for t in Meses}

# Costos
CB = {1: 8000, 2: 8000, 3: 8000, 4: 8000, 5: 8000, 6: 8000}
CP = {1: 24000, 2: 24000, 3: 24000, 4: 24000, 5: 24000, 6: 24000}
CI = 2500
CW = 2864325
CH = 625000
CL = 500000

#Parámetros
Demanda = {1: 10680, 2: 9600, 3: 11200, 4: 13100, 5: 8900, 6: 10320}
Dias = {1: 21, 2: 20, 3: 20,4: 21, 5: 22, 6: 23}

# Función Objetivo
modelo += lp.lpSum(CP[t] * P[t]+ CB[t] * B[t] + CI * I[t] + CW * W[t] + CH * H[t] + CL * L[t] for t in Meses)

# Restricciones

# Balance de Recursos Para t = 1
modelo += W[1] - H[1] + L[1] == 85

# Balance de Recursos Para t > 1
for t in Meses:
    if t > 1:
        modelo += W[t] - W[t-1] - H[t] + L[t] == 0

# Balance de inventario para t = 1
modelo += I[1] - B[1] - P[1] == -Demanda[1]

# Balance de inventario para t > 1
for t in Meses:
    if t > 1:
        modelo += I[t] - B[t] - I[t-1] + B[t-1] - P[t] == -Demanda[t]

# Capacidad
for t in Meses:
    modelo += P[t] <= 6 * Dias[t] * W[t]

# Resolver el modelo
resultado = modelo.solve(lp.COIN_CMD(msg=False))

# Resultados
print("Estado de la solución:", resultado.status_str)

if resultado.status == lp.LpSolveStatus.Optimal:

    for v in modelo.variables():
        print(f"{v.name} = {v.varValue}")

    print("Costo total = $", lp.value(modelo.objective))

else:
    print("No se encontró una solución óptima.")


Estado de la solución: Optimal
P1 = 10584.0
P2 = 9600.0
P3 = 9600.0
P4 = 10080.0
P5 = 0.0
P6 = 0.0
B1 = 96.0
B2 = 96.0
B3 = 1696.0
B4 = 4716.0
B5 = 13616.0
B6 = 23936.0
I1 = 0.0
I2 = 0.0
I3 = 0.0
I4 = 0.0
I5 = 0.0
I6 = 0.0
W1 = 84.0
W2 = 80.0
W3 = 80.0
W4 = 80.0
W5 = 80.0
W6 = 80.0
H1 = 0.0
H2 = 0.0
H3 = 0.0
H4 = 0.0
H5 = 0.0
H6 = 0.0
L1 = 1.0
L2 = 4.0
L3 = 0.0
L4 = 0.0
L5 = 0.0
L6 = 0.0
Costo total = $ 2698817300.0
